In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install split-folders

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from transformers import ViTForImageClassification, ViTFeatureExtractor
from tqdm import tqdm

In [3]:
# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [4]:
import splitfolders

input_folder = "/kaggle/input/sugarcane-plant-diseases-dataset/Sugarcane_leafs"
splitfolders.ratio(input_folder, output="/kaggle/working/sugarcanedisease", seed=42, ratio=(0.7, 0.15, 0.15), group_prefix=None)

Copying files: 19926 files [03:36, 92.03 files/s] 


In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (256, 256)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=90,
                                   horizontal_flip=True,
                                   vertical_flip=True,
                                   zoom_range=0.2)

val_test_datagen = ImageDataGenerator(rescale=1./255)


#----------------------------------------

train_generator = train_datagen.flow_from_directory('/kaggle/working/sugarcanedisease/train', 
                                                    target_size=IMG_SIZE, 
                                                    batch_size=BATCH_SIZE, 
                                                    class_mode='categorical')

val_generator = val_test_datagen.flow_from_directory('/kaggle/working/sugarcanedisease/val', 
                                                target_size=IMG_SIZE, 
                                                batch_size=BATCH_SIZE, 
                                                class_mode='categorical')

test_generator = val_test_datagen.flow_from_directory('/kaggle/working/sugarcanedisease/test', 
                                                  target_size=IMG_SIZE, 
                                                  batch_size=BATCH_SIZE, 
                                                  class_mode='categorical', 
                                                  shuffle=False)

Found 13946 images belonging to 6 classes.
Found 2986 images belonging to 6 classes.
Found 2994 images belonging to 6 classes.


In [6]:
import os  
import torch  
from torchvision import datasets, transforms  
  
# Define data directories  
data_dir = '/kaggle/working/sugarcanedisease'  
train_dir = os.path.join(data_dir, 'train')  
val_dir = os.path.join(data_dir, 'val')  
test_dir = os.path.join(data_dir, 'test')  
  
# Define image size and batch size  
IMG_SIZE = (224, 224)  
BATCH_SIZE = 16  
  
# Define data transformations  
# Corresponding to Keras ImageDataGenerator parameters  
data_transforms = {  
    'train': transforms.Compose([  
        transforms.Resize(IMG_SIZE),  
        transforms.RandomRotation(90),  
        transforms.RandomHorizontalFlip(),  
        transforms.RandomVerticalFlip(),  
        transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),  
        transforms.ToTensor(),  
        # No need for rescale=1./255 because ToTensor() does that  
    ]),  
    'val': transforms.Compose([  
        transforms.Resize(IMG_SIZE),  
        transforms.ToTensor(),  
        # No need for rescale=1./255 because ToTensor() does that  
    ]),  
    'test': transforms.Compose([  
        transforms.Resize(IMG_SIZE),  
        transforms.ToTensor(),  
        # No need for rescale=1./255 because ToTensor() does that  
    ]),  
}  
  
# Create datasets using ImageFolder  
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms['train'])  
val_dataset = datasets.ImageFolder(val_dir, transform=data_transforms['val'])  
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms['test'])  
  
# Create DataLoaders  
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)  
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)  
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4) 

In [7]:
# Compute class counts for the training dataset  
from collections import Counter  

In [8]:
# Get all the labels from the training dataset  
labels = [sample[1] for sample in train_dataset.samples]  
class_counts = Counter(labels)  
print("Class counts:", class_counts) 

Class counts: Counter({0: 3360, 1: 2192, 3: 2175, 4: 2158, 5: 2121, 2: 1940})


In [9]:
# Get total number of samples and number of classes  
total_samples = len(train_dataset)  
num_classes = len(train_dataset.classes)  

In [10]:
# Compute class weights inversely proportional to class frequencies  
class_weights = []  
for i in range(num_classes):  
    class_weight = total_samples / (num_classes * class_counts[i])  
    class_weights.append(class_weight)  

In [11]:
# Convert class weights to a tensor and move to device  
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)  
print("Class weights:", class_weights)  

Class weights: tensor([0.6918, 1.0604, 1.1981, 1.0687, 1.0771, 1.0959], device='cuda:0')


In [12]:
# Create DataLoaders  
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)  
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)  
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)  

In [13]:
# Print dataset sizes  
print(f'Training dataset size: {len(train_dataset)}')  
print(f'Validation dataset size: {len(val_dataset)}')  
print(f'Test dataset size: {len(test_dataset)}')  

Training dataset size: 13946
Validation dataset size: 2986
Test dataset size: 2994


In [14]:
# Print class names  
print("Classes:", train_dataset.classes)  
num_classes = len(train_dataset.classes)

Classes: ['BacterialBlights', 'Healthy', 'Mosaic', 'RedRot', 'Rust', 'Yellow']


In [15]:
# Load the Swin Transformer model  
from transformers import SwinForImageClassification, AutoConfig  

In [16]:
# Choose the pre-trained Swin model  
model_name = 'microsoft/swin-base-patch4-window7-224'  

In [17]:
# Load the pre-trained model configuration  
config = AutoConfig.from_pretrained(model_name)  
config.num_labels = num_classes  # Set the number of output labels  

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

In [18]:
# Load the Swin Transformer model  
model = SwinForImageClassification.from_pretrained(model_name, config=config, ignore_mismatched_sizes=True)  

model.safetensors:   0%|          | 0.00/352M [00:00<?, ?B/s]

Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-base-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([6]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([6, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
# Move the model to the device (GPU or CPU)  
model.to(device)  

SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): SwinAttention(
                (self): SwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )
                (output): SwinSelfOutput(

In [20]:
# Define the loss function with class weights  
criterion = nn.CrossEntropyLoss(weight=class_weights)  

In [21]:
# Use only trainable parameters for the optimizer  
optimizer = optim.Adam(model.parameters(), lr=1e-4)  

In [29]:
# Define a learning rate scheduler (optional)  
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)  

In [22]:
# Training function  
def train(model, dataloader, optimizer, criterion):  
    model.train()  
    running_loss = 0.0  
    correct = 0  
    total = 0  
  
    for images, labels in tqdm(dataloader, desc='Training'):  
        images = images.to(device)  
        labels = labels.to(device)  
  
        optimizer.zero_grad()  
  
        # Forward pass  
        outputs = model(images)  
        logits = outputs.logits  
        loss = criterion(logits, labels)  
  
        # Backward pass and optimization  
        loss.backward()  
        optimizer.step()  
  
        running_loss += loss.item() * images.size(0)  
        _, preds = torch.max(logits, 1)  
        total += labels.size(0)  
        correct += (preds == labels).sum().item()  
  
    epoch_loss = running_loss / len(dataloader.dataset)  
    epoch_acc = correct / total  
    return epoch_loss, epoch_acc  
  
# Validation function  
def validate(model, dataloader, criterion):  
    model.eval()  
    running_loss = 0.0  
    correct = 0  
    total = 0  
  
    with torch.no_grad():  
        for images, labels in tqdm(dataloader, desc='Validation'):  
            images = images.to(device)  
            labels = labels.to(device)  
  
            outputs = model(images)  
            logits = outputs.logits  
            loss = criterion(logits, labels)  
  
            running_loss += loss.item() * images.size(0)  
            _, preds = torch.max(logits, 1)  
            total += labels.size(0)  
            correct += (preds == labels).sum().item()  
  
    epoch_loss = running_loss / len(dataloader.dataset)  
    epoch_acc = correct / total  
    return epoch_loss, epoch_acc  
  


In [23]:
# Training Loop  
num_epochs = 10  
train_losses = []  
val_losses = []  
train_accuracies = []  
val_accuracies = []  
  
for epoch in range(num_epochs):  
    print(f'Epoch {epoch+1}/{num_epochs}')  
  
    # Train the model  
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)  
    train_losses.append(train_loss)  
    train_accuracies.append(train_acc)  
  
    print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')  
  
    # Validate the model  
    val_loss, val_acc = validate(model, val_loader, criterion)  
    val_losses.append(val_loss)  
    val_accuracies.append(val_acc)  
  
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')  
  
    # Step the scheduler (if defined)  
    scheduler.step()  
  
    print('-' * 30)  
  


Epoch 1/10


Training: 100%|██████████| 872/872 [04:56<00:00,  2.94it/s]


Train Loss: 0.4368, Train Acc: 0.8474


Validation: 100%|██████████| 187/187 [00:21<00:00,  8.71it/s]


Val Loss: 0.2701, Val Acc: 0.9190


NameError: name 'scheduler' is not defined

In [ ]:
# Plot training and validation loss  
plt.figure(figsize=(10, 5))  
plt.plot(train_losses, label='Train Loss')  
plt.plot(val_losses, label='Val Loss')  
plt.legend()  
plt.title('Loss')  
plt.show()  
  
# Plot training and validation accuracy  
plt.figure(figsize=(10, 5))  
plt.plot(train_accuracies, label='Train Accuracy')  
plt.plot(val_accuracies, label='Val Accuracy')  
plt.legend()  
plt.title('Accuracy')  
plt.show

In [ ]:
# Plot confusion matrix and classification report  
from sklearn.metrics import confusion_matrix, classification_report  
import seaborn as sns  
  
# Collect all the true labels and predicted labels for the validation set  
all_labels = []  
all_preds = []  
  
model.eval()  
with torch.no_grad():  
    for images, labels in tqdm(val_loader, desc='Collecting predictions'):  
        images = images.to(device)  
        labels = labels.to(device)  
        outputs = model(images)  
        logits = outputs.logits  
        _, preds = torch.max(logits, 1)  
        all_labels.extend(labels.cpu().numpy())  
        all_preds.extend(preds.cpu().numpy())  
  
# Calculate the confusion matrix  
cm = confusion_matrix(all_labels, all_preds)  
  
# Plot the confusion matrix  
plt.figure(figsize=(12, 10))  
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=train_dataset.classes, yticklabels=train_dataset.classes)  
plt.xlabel('Predicted labels')  
plt.ylabel('True labels')  
plt.title('Confusion Matrix')  
plt.show()  

In [ ]:
# Print classification report  
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes)) 

In [ ]:
# Save the fine-tuned model weights  
torch.save(model.state_dict(), '/kaggle/working/swin_sugarcane_disease_detection.pth')  

#Batch Size 8

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from transformers import ViTForImageClassification, ViTFeatureExtractor
from tqdm import tqdm

In [4]:
# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [5]:
import splitfolders

input_folder = "/kaggle/input/sugarcane-plant-diseases-dataset/Sugarcane_leafs"
splitfolders.ratio(input_folder, output="/kaggle/working/sugarcanedisease", seed=42, ratio=(0.7, 0.15, 0.15), group_prefix=None)

Copying files: 19926 files [03:18, 100.49 files/s]


In [6]:
import os  
import torch  
from torchvision import datasets, transforms  
  
# Define data directories  
data_dir = '/kaggle/working/sugarcanedisease'  
train_dir = os.path.join(data_dir, 'train')  
val_dir = os.path.join(data_dir, 'val')  
test_dir = os.path.join(data_dir, 'test')  
  
# Define image size and batch size  
IMG_SIZE = (224, 224)  
BATCH_SIZE = 8  
  
# Define data transformations  
# Corresponding to Keras ImageDataGenerator parameters  
data_transforms = {  
    'train': transforms.Compose([  
        transforms.Resize(IMG_SIZE),  
        transforms.RandomRotation(90),  
        transforms.RandomHorizontalFlip(),  
        transforms.RandomVerticalFlip(),  
        transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),  
        transforms.ToTensor(),  
        # No need for rescale=1./255 because ToTensor() does that  
    ]),  
    'val': transforms.Compose([  
        transforms.Resize(IMG_SIZE),  
        transforms.ToTensor(),  
        # No need for rescale=1./255 because ToTensor() does that  
    ]),  
    'test': transforms.Compose([  
        transforms.Resize(IMG_SIZE),  
        transforms.ToTensor(),  
        # No need for rescale=1./255 because ToTensor() does that  
    ]),  
}  
  
# Create datasets using ImageFolder  
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms['train'])  
val_dataset = datasets.ImageFolder(val_dir, transform=data_transforms['val'])  
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms['test'])  
  
# Create DataLoaders  
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)  
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)  
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4) 

In [7]:
# Get total number of samples and number of classes  
total_samples = len(train_dataset)  
num_classes = len(train_dataset.classes)  

In [9]:
# Create DataLoaders  
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)  
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)  
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)  

In [10]:
# Load the Swin Transformer model  
from transformers import SwinForImageClassification, AutoConfig  

In [11]:
# Choose the pre-trained Swin model  
model_name = 'microsoft/swin-base-patch4-window7-224'  

In [12]:
# Print class names  
print("Classes:", train_dataset.classes)  
num_classes = len(train_dataset.classes)

Classes: ['BacterialBlights', 'Healthy', 'Mosaic', 'RedRot', 'Rust', 'Yellow']


In [13]:
# Load the pre-trained model configuration  
config = AutoConfig.from_pretrained(model_name)  
config.num_labels = num_classes  # Set the number of output labels  

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

In [14]:
# Load the Swin Transformer model  
model = SwinForImageClassification.from_pretrained(model_name, config=config, ignore_mismatched_sizes=True)  

model.safetensors:   0%|          | 0.00/352M [00:00<?, ?B/s]

Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-base-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([6]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 1024]) in the checkpoint and torch.Size([6, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
# Move the model to the device (GPU or CPU)  
model.to(device) 

SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): SwinAttention(
                (self): SwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )
                (output): SwinSelfOutput(

In [16]:
# Define the loss function with class weights  
criterion = nn.CrossEntropyLoss(weight=class_weights)  

TypeError: cannot assign 'list' object to buffer 'weight' (torch Tensor or None required)

In [ ]:
# Use only trainable parameters for the optimizer  
optimizer = optim.Adam(model.parameters(), lr=1e-4)  

In [ ]:
# Define a learning rate scheduler (optional)  
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)  

In [ ]:
# Training function  
def train(model, dataloader, optimizer, criterion):  
    model.train()  
    running_loss = 0.0  
    correct = 0  
    total = 0  
  
    for images, labels in tqdm(dataloader, desc='Training'):  
        images = images.to(device)  
        labels = labels.to(device)  
  
        optimizer.zero_grad()  
  
        # Forward pass  
        outputs = model(images)  
        logits = outputs.logits  
        loss = criterion(logits, labels)  
  
        # Backward pass and optimization  
        loss.backward()  
        optimizer.step()  
  
        running_loss += loss.item() * images.size(0)  
        _, preds = torch.max(logits, 1)  
        total += labels.size(0)  
        correct += (preds == labels).sum().item()  
  
    epoch_loss = running_loss / len(dataloader.dataset)  
    epoch_acc = correct / total  
    return epoch_loss, epoch_acc  
  
# Validation function  
def validate(model, dataloader, criterion):  
    model.eval()  
    running_loss = 0.0  
    correct = 0  
    total = 0  
  
    with torch.no_grad():  
        for images, labels in tqdm(dataloader, desc='Validation'):  
            images = images.to(device)  
            labels = labels.to(device)  
  
            outputs = model(images)  
            logits = outputs.logits  
            loss = criterion(logits, labels)  
  
            running_loss += loss.item() * images.size(0)  
            _, preds = torch.max(logits, 1)  
            total += labels.size(0)  
            correct += (preds == labels).sum().item()  
  
    epoch_loss = running_loss / len(dataloader.dataset)  
    epoch_acc = correct / total  
    return epoch_loss, epoch_acc  
  